# Excel vs ЦФТ: топ расхождений (Красноярский / Калининградский, апр–июн 2026)

**Цель:** по 5 `agr_id` на каждый РФ × тариф с **максимальным** расхождением `commission_monthly`.

**Тарифы:** только `0`, `По Акту индивидуальный`, `Стандарт` (без «Меню возможностей»).

**НДС:** в озере комиссия **с НДС 22%**, в Excel — **без НДС**.
Ожидание озера = `excel × 1.22`. Если `lake ≈ excel×1.22` — это **не** расхождение.

**Озеро:** SQL коллеги `DOG_OPER` → ранг по `sum(c_calc_summ)`; `c_pay_summ` в выгрузке рядом.

Выход: `/home/jovyan/documents/Equaring/Data/rf_excel_vs_cft_krasn_kalin_apr_jun/`


In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))


In [ ]:
DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
OUT_DIR = DATA_DIR / 'rf_excel_vs_cft_krasn_kalin_apr_jun'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_RFS = ['Красноярский', 'Калининградский']
VAT_FACTOR = 1.22
COMPARE_ABS_TOL = 0.01
TOP_N = 5
MONTHS = ['2026-04', '2026-05', '2026-06']
TARIFF_ORDER = ['0', 'По Акту индивидуальный', 'Стандарт']
MONTH_COL_RU = {'2026-04': 'апрель', '2026-05': 'май', '2026-06': 'июнь'}

excel_sources = [
    {'report_month': '2026-04-01', 'path': DATA_DIR / '04_Апрель_2026.xlsx', 'header': 0},
    {'report_month': '2026-05-01', 'path': DATA_DIR / '05_Май_2026.xlsx', 'header': 0},
    {'report_month': '2026-06-01', 'path': DATA_DIR / '06_Июнь_2026.xlsx', 'header': 0},
]

def rf_slug(name: str) -> str:
    mapping = {
        'Красноярский': 'krasnoyarsk',
        'Калининградский': 'kaliningrad',
    }
    return mapping.get(name, re.sub(r'[^a-zA-Z0-9а-яА-Я]+', '_', name).strip('_').lower())

for src in excel_sources:
    p = Path(src['path'])
    print(f"{src['report_month'][:7]}: exists={p.exists()} | {p}")
print('OUT_DIR:', OUT_DIR)
print('TARGET_RFS:', TARGET_RFS)
print('VAT_FACTOR:', VAT_FACTOR, '| TOP_N:', TOP_N)
print('TARIFF_ORDER:', TARIFF_ORDER)


## 1) Helpers + загрузка Excel


In [ ]:
def normalize_colname(value):
    s = str(value).lower().replace('\n', ' ').replace('\r', ' ').replace('\xa0', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    s = s.replace('₽', 'руб').replace('%', 'pct')
    s = re.sub(r'[^a-zа-я0-9]+', '', s)
    return s


def pick_column(columns, aliases):
    cols = list(columns)
    norm_map = {normalize_colname(c): c for c in cols}
    for alias in aliases:
        if alias in cols:
            return alias
        key = normalize_colname(alias)
        if key in norm_map:
            return norm_map[key]
    for alias in aliases:
        key = normalize_colname(alias)
        for nk, original in norm_map.items():
            if key and key in nk:
                return original
    return None


def to_num(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace('\xa0', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace(',', '.', regex=False),
        errors='coerce',
    )


def normalize_agr_id(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().replace('\xa0', '').replace(' ', '')
    if s.endswith('.0'):
        s = s[:-2]
    if re.fullmatch(r'\d+', s):
        return s
    return s or np.nan


def normalize_inn(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().replace('\xa0', '').replace(' ', '')
    if s.endswith('.0'):
        s = s[:-2]
    s = re.sub(r'\D', '', s)
    return s or np.nan


def classify_tariff(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    sl = s.lower()
    if s == '0' or sl in {'0', '0.0', 'zero'}:
        return '0'
    if 'меню возможностей' in sl or sl.startswith('меню'):
        return 'Меню возможностей'  # исключаем позже
    if 'по акту' in sl or 'индивидуальн' in sl:
        return 'По Акту индивидуальный'
    if 'стандарт' in sl:
        return 'Стандарт'
    return None


COLUMN_ALIASES = {
    'agr_id': ['ID договора', 'agr_id', 'ИД договора', 'id договора'],
    'company_name': ['Наименование', 'Наименование клиента', 'company_name', 'Клиент'],
    'inn': ['ИНН', 'ИНН клиента', 'inn'],
    'contract_number': ['Номер договора', 'contract_number', '№ договора'],
    'd_valid_from': ['Дата регистрации договора', 'Дата заключения', 'd_valid_from', 'Дата регистрации'],
    'd_valid_to': ['Дата закрытия договора', 'Дата закрытия', 'd_valid_to', 'Дата расторжения'],
    'tariff': ['Тариф', 'tariff', 'Тарифный план'],
    'trx_sum': ['Сумма операций', 'trx_sum', 'Оборот'],
    'commission_from_ops': ['Комиссия (% с операций)', 'Комиссия % с операций', 'commission_from_ops', 'acq_pct'],
    'commission_monthly': ['Комиссия (₽ в месяц)', 'Комиссия (руб в месяц)', 'commission_monthly', 'Комиссия в месяц'],
    'filial': ['Филиал', 'Региональный филиал', 'Филиал договора', 'branch_nm', 'filial_rf'],
}


def resolve_columns(raw):
    resolved = {}
    for key, aliases in COLUMN_ALIASES.items():
        col = pick_column(raw.columns, aliases)
        if col is None and key not in ('d_valid_to', 'commission_from_ops', 'contract_number'):
            raise KeyError(f'Не найдена колонка {key}: aliases={aliases} | cols={list(raw.columns)[:40]}')
        resolved[key] = col
    return resolved


def load_excel_all():
    frames = []
    resolved_by_month = []
    for src in excel_sources:
        raw = pd.read_excel(src['path'], header=src['header'])
        resolved = resolve_columns(raw)
        df = pd.DataFrame({
            'agr_id': raw[resolved['agr_id']].map(normalize_agr_id),
            'company_name': raw[resolved['company_name']],
            'inn': raw[resolved['inn']].map(normalize_inn) if resolved['inn'] else np.nan,
            'contract_number': raw[resolved['contract_number']] if resolved['contract_number'] else np.nan,
            'd_valid_from': pd.to_datetime(raw[resolved['d_valid_from']], errors='coerce'),
            'd_valid_to': (
                pd.to_datetime(raw[resolved['d_valid_to']], errors='coerce')
                if resolved['d_valid_to'] else pd.NaT
            ),
            'tariff_raw': raw[resolved['tariff']],
            'trx_sum': to_num(raw[resolved['trx_sum']]),
            'commission_from_ops': (
                to_num(raw[resolved['commission_from_ops']])
                if resolved['commission_from_ops'] else np.nan
            ),
            'commission_monthly': to_num(raw[resolved['commission_monthly']]),
            'filial_raw': raw[resolved['filial']],
        })
        df['report_month'] = pd.to_datetime(src['report_month'])
        df['report_month_str'] = df['report_month'].dt.strftime('%Y-%m')
        df['tariff_group'] = df['tariff_raw'].map(classify_tariff)
        df['filial_norm'] = (
            df['filial_raw'].astype(str)
            .str.replace('\xa0', ' ', regex=False)
            .str.replace(r'\s+', ' ', regex=True)
            .str.strip()
        )
        df['filial_rf'] = df['filial_norm'].map(
            lambda v: re.match(r'^(.*?РФ)', v).group(1) if isinstance(v, str) and 'РФ' in v else v
        )
        frames.append(df)
        resolved_by_month.append({
            'month': src['report_month'][:7],
            **{k: v for k, v in resolved.items()},
            'rows_raw': len(raw),
        })
        print(f"{src['report_month'][:7]}: raw={len(raw):,}")
    excel_all = pd.concat(frames, ignore_index=True)
    display(pd.DataFrame(resolved_by_month))
    print('Total Excel rows:', len(excel_all))
    return excel_all


excel_all = load_excel_all()
print('Sample filial_rf:')
display(excel_all['filial_rf'].value_counts(dropna=False).head(30))
print('Tariff groups (raw):')
display(excel_all['tariff_group'].value_counts(dropna=False))


## 2) Кандидаты по РФ (без SPB-фильтров)

Стабильный тариф во всех 3 месяцах; только `0` / `По Акту индивидуальный` / `Стандарт`.


In [ ]:
def candidates_for_rf(target_rf: str) -> pd.DataFrame:
    mask_rf = excel_all['filial_norm'].str.contains(target_rf, case=False, na=False)
    excel_rf = excel_all[mask_rf & excel_all['agr_id'].notna()].copy()
    excel_rf = excel_rf[excel_rf['tariff_group'].isin(TARIFF_ORDER)].copy()
    print(f'\n######## RF={target_rf} | rows={len(excel_rf):,} | agr={excel_rf["agr_id"].nunique():,} ########')
    display(excel_rf['tariff_group'].value_counts(dropna=False))

    months_per_agr = (
        excel_rf.groupby('agr_id')['report_month_str'].nunique().rename('months_cnt').reset_index()
    )
    stable_ids = set(months_per_agr.loc[months_per_agr['months_cnt'] == len(MONTHS), 'agr_id'])
    stable_df = excel_rf[excel_rf['agr_id'].isin(stable_ids)].copy()

    group_nunique = stable_df.groupby('agr_id')['tariff_group'].nunique(dropna=True)
    stable_tariff_ids = set(group_nunique[group_nunique == 1].index)
    has_group = (
        stable_df[stable_df['agr_id'].isin(stable_tariff_ids)]
        .groupby('agr_id')['tariff_group'].first()
    )
    has_group = has_group[has_group.isin(TARIFF_ORDER)]
    candidate_df = stable_df[stable_df['agr_id'].isin(set(has_group.index))].copy()
    candidate_df['tariff_group'] = candidate_df['agr_id'].map(has_group)
    candidate_df['target_rf'] = target_rf

    print(f'Во всех {len(MONTHS)} месяцах: {len(stable_ids):,}')
    print(f'Стабильный целевой tariff_group: {candidate_df["agr_id"].nunique():,}')
    display(
        candidate_df.drop_duplicates('agr_id')
        .groupby('tariff_group', as_index=False)['agr_id'].nunique()
        .rename(columns={'agr_id': 'agr_id_cnt'})
    )
    return candidate_df


candidate_parts = [candidates_for_rf(rf) for rf in TARGET_RFS]
candidates_df = pd.concat(candidate_parts, ignore_index=True) if candidate_parts else pd.DataFrame()
ALL_CANDIDATE_IDS = sorted({str(a) for a in candidates_df['agr_id'].dropna().unique()})
print('ALL candidate agr_id:', len(ALL_CANDIDATE_IDS))
if not ALL_CANDIDATE_IDS:
    raise RuntimeError('Нет кандидатов — проверьте Excel / названия РФ / тарифы.')


## 3) Impala + DOG_OPER по всем кандидатам

`ods.scd1_z_R2_IP_DOG_OPER` + `VID_COMISS` + merchants + client, `c_date_create > '2026-03-31'`.


In [ ]:
from rail_connectors.connection import connect

if 'imp' not in globals() or imp is None:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': 'Shestopalov-VYur'},
    )
    imp._init_connection()
    print('Impala connected')
else:
    print('Reuse existing imp connection')

agr_ids_sql = ', '.join(str(int(x)) if str(x).isdigit() else f"'{x}'" for x in ALL_CANDIDATE_IDS)
print('agr_ids_sql count:', len(ALL_CANDIDATE_IDS))


In [ ]:
sql_commissions = f'''
select distinct
  m.c_cl_org cft_id,
  cl.c_name name_org,
  m.id id_agreement,
  m.c_name_in_pr agreement_num,
  m.c_date_begin,
  vc.id,
  vc.c_name commis_type,
  o.c_date_create,
  o.c_pay_summ,
  o.c_calc_summ
from ods.scd1_z_R2_IP_DOG_OPER o
join ods.scd1_z_R2_VID_COMISS vc on vc.id = o.c_vid_comiss
join ods.scd1_z_r2_ip_merchants m on m.id = o.c_parent_id
join ods.scd1_z_client cl on m.c_cl_org = cl.id
where o.c_parent_class = 'R2_IP_MERCHANTS'
  and cl.class_id = 'CL_ORG'
  and m.id in ({agr_ids_sql})
  and o.c_date_create > '2026-03-31'
order by o.c_date_create desc
'''

print('SQL length chars:', len(sql_commissions))
print(sql_commissions[:800], '...')

with imp:
    imp.execute('set MEM_LIMIT=8g')
    raw_df = imp.fetch(sql_commissions)

if raw_df is None:
    raw_df = pd.DataFrame()

print(f'raw rows: {len(raw_df):,}')
display(raw_df.head(30))


### Запасной SQL (lower-case)

Только если предыдущая ячейка упала с «table not found».


In [ ]:
RUN_LOWERCASE_FALLBACK = False

if RUN_LOWERCASE_FALLBACK:
    sql_commissions_lc = f'''
    select distinct
      m.c_cl_org cft_id,
      cl.c_name name_org,
      m.id id_agreement,
      m.c_name_in_pr agreement_num,
      m.c_date_begin,
      vc.id,
      vc.c_name commis_type,
      o.c_date_create,
      o.c_pay_summ,
      o.c_calc_summ
    from ods.scd1_z_r2_ip_dog_oper o
    join ods.scd1_z_r2_vid_comiss vc on vc.id = o.c_vid_comiss
    join ods.scd1_z_r2_ip_merchants m on m.id = o.c_parent_id
    join ods.scd1_z_client cl on m.c_cl_org = cl.id
    where o.c_parent_class = 'R2_IP_MERCHANTS'
      and cl.class_id = 'CL_ORG'
      and m.id in ({agr_ids_sql})
      and o.c_date_create > '2026-03-31'
    order by o.c_date_create desc
    '''
    with imp:
        imp.execute('set MEM_LIMIT=8g')
        raw_df = imp.fetch(sql_commissions_lc)
    if raw_df is None:
        raw_df = pd.DataFrame()
    print(f'raw rows (lowercase): {len(raw_df):,}')
else:
    print('SKIP lowercase fallback')


In [ ]:
# Агрегация кандидатов: id_agreement × месяц
if raw_df.empty:
    by_month_long = pd.DataFrame(columns=[
        'id_agreement', 'month', 'month_ru', 'rows', 'c_pay_summ', 'c_calc_summ'
    ])
    print('WARN: raw_df пустой')
else:
    agg_src = raw_df.copy()
    agg_src['id_agreement'] = agg_src['id_agreement'].astype(str).str.replace(r'\.0$', '', regex=True)
    agg_src['c_date_create'] = pd.to_datetime(agg_src['c_date_create'], errors='coerce')
    agg_src['month'] = agg_src['c_date_create'].dt.strftime('%Y-%m')
    agg_src = agg_src[agg_src['month'].isin(MONTHS)].copy()
    agg_src['month_ru'] = agg_src['month'].map(MONTH_COL_RU)
    agg_src['c_pay_summ'] = pd.to_numeric(agg_src['c_pay_summ'], errors='coerce')
    agg_src['c_calc_summ'] = pd.to_numeric(agg_src['c_calc_summ'], errors='coerce')
    by_month_long = (
        agg_src.groupby(['id_agreement', 'month', 'month_ru'], as_index=False)
        .agg(
            rows=('c_date_create', 'size'),
            c_pay_summ=('c_pay_summ', 'sum'),
            c_calc_summ=('c_calc_summ', 'sum'),
        )
        .sort_values(['id_agreement', 'month'])
        .reset_index(drop=True)
    )
    print('by_month_long rows:', len(by_month_long))
    display(by_month_long.head(40))

out_raw_cand = OUT_DIR / 'lake_dog_oper_commissions_candidates_raw.xlsx'
out_month_cand = OUT_DIR / 'lake_dog_oper_commissions_candidates_by_month.xlsx'
raw_df.to_excel(out_raw_cand, index=False)
by_month_long.to_excel(out_month_cand, index=False)
print('Saved:', out_raw_cand)
print('Saved:', out_month_cand)


## 4) Сверка с НДС + TOP-5 на РФ × тариф

- `excel_gross = excel_commission_monthly * 1.22`
- `delta_calc = cft_calc_summ - excel_gross`
- `score(agr_id) = max(|delta_calc|)` по месяцам
- TOP-5 по score для каждой пары `(target_rf, tariff_group)`


In [ ]:
# Excel month grain
excel_long = candidates_df[[
    'target_rf', 'agr_id', 'report_month_str', 'commission_monthly', 'trx_sum',
    'tariff_group', 'tariff_raw', 'company_name', 'inn', 'filial_rf',
]].copy()
excel_long['agr_id'] = excel_long['agr_id'].astype(str)
excel_month = (
    excel_long.groupby(['target_rf', 'agr_id', 'report_month_str'], as_index=False)
    .agg(
        excel_commission_monthly=('commission_monthly', 'sum'),
        excel_trx_sum=('trx_sum', 'sum'),
        tariff_group=('tariff_group', 'first'),
        tariff_raw=('tariff_raw', 'first'),
        company_name=('company_name', 'first'),
        inn=('inn', 'first'),
        filial_rf=('filial_rf', 'first'),
    )
)

if by_month_long.empty:
    cft_month = pd.DataFrame(columns=[
        'agr_id', 'report_month_str', 'cft_rows', 'cft_pay_summ', 'cft_calc_summ'
    ])
else:
    cft_month = by_month_long.rename(columns={
        'id_agreement': 'agr_id',
        'month': 'report_month_str',
        'rows': 'cft_rows',
        'c_pay_summ': 'cft_pay_summ',
        'c_calc_summ': 'cft_calc_summ',
    })[['agr_id', 'report_month_str', 'cft_rows', 'cft_pay_summ', 'cft_calc_summ']].copy()
    cft_month['agr_id'] = cft_month['agr_id'].astype(str)

compare_df = excel_month.merge(cft_month, on=['agr_id', 'report_month_str'], how='left')
compare_df['cft_rows'] = compare_df['cft_rows'].fillna(0).astype(int)
compare_df['cft_pay_summ'] = pd.to_numeric(compare_df['cft_pay_summ'], errors='coerce').fillna(0.0)
compare_df['cft_calc_summ'] = pd.to_numeric(compare_df['cft_calc_summ'], errors='coerce').fillna(0.0)
compare_df['excel_commission_monthly'] = pd.to_numeric(
    compare_df['excel_commission_monthly'], errors='coerce'
).fillna(0.0)

compare_df['excel_gross'] = compare_df['excel_commission_monthly'] * VAT_FACTOR
compare_df['delta_calc'] = compare_df['cft_calc_summ'] - compare_df['excel_gross']
compare_df['delta_pay'] = compare_df['cft_pay_summ'] - compare_df['excel_gross']
compare_df['abs_delta_calc'] = compare_df['delta_calc'].abs()
compare_df['missing_in_cft'] = compare_df['cft_rows'] == 0
compare_df['is_discrepancy'] = compare_df['abs_delta_calc'] > COMPARE_ABS_TOL

# score per agr_id within RF
agr_score = (
    compare_df.groupby(['target_rf', 'tariff_group', 'agr_id'], as_index=False)
    .agg(
        score=('abs_delta_calc', 'max'),
        disc_months=('is_discrepancy', 'sum'),
        missing_months=('missing_in_cft', 'sum'),
        company_name=('company_name', 'first'),
        inn=('inn', 'first'),
        tariff_raw=('tariff_raw', 'first'),
        filial_rf=('filial_rf', 'first'),
    )
)

top_rows = []
for rf in TARGET_RFS:
    for tariff in TARIFF_ORDER:
        pool = agr_score[(agr_score['target_rf'] == rf) & (agr_score['tariff_group'] == tariff)].copy()
        # сначала реальные расхождения (score > tol), потом добиваем остальными
        pool = pool.sort_values(['score', 'agr_id'], ascending=[False, True])
        disc_pool = pool[pool['score'] > COMPARE_ABS_TOL]
        if len(disc_pool) >= TOP_N:
            pool = disc_pool.head(TOP_N)
        else:
            rest = pool[~pool['agr_id'].isin(disc_pool['agr_id'])]
            pool = pd.concat([disc_pool, rest], ignore_index=True).head(TOP_N)
            if len(disc_pool) == 0:
                print(f'WARN: {rf} | {tariff}: нет расхождений после НДС — TOP-{TOP_N} заполнен совпадающими agr_id')
        pool = pool.copy()
        pool['rank'] = range(1, len(pool) + 1)
        top_rows.append(pool)
        n_cand = len(agr_score[(agr_score['target_rf'] == rf) & (agr_score['tariff_group'] == tariff)])
        print(
            f'{rf} | {tariff}: candidates={n_cand}, disc={len(disc_pool)}, '
            f'top={len(pool)}, max_score={pool["score"].max() if len(pool) else 0:.2f}'
        )

top_agr_df = pd.concat(top_rows, ignore_index=True) if top_rows else pd.DataFrame()
SELECTED_IDS = sorted(set(top_agr_df['agr_id'].astype(str)))
print('SELECTED top agr_id:', len(SELECTED_IDS))
display(top_agr_df)

top_compare = compare_df.merge(
    top_agr_df[['target_rf', 'tariff_group', 'agr_id', 'score', 'rank']],
    on=['target_rf', 'tariff_group', 'agr_id'],
    how='inner',
).sort_values(['target_rf', 'tariff_group', 'rank', 'report_month_str']).reset_index(drop=True)

display(top_compare.head(60))


## 5) Итоговая таблица: 5 agr_id × тариф × 3 месяца (по каждому РФ)

Колонки:
- `agr_id`
- `excel_commission` — сумма комиссии в Excel (без НДС)
- `lake_commission` — сумма комиссии в озере (`c_calc_summ`, с НДС)

Одни и те же TOP-5 `agr_id` на тариф внутри РФ повторяются на каждый из месяцев апр/май/июнь.


In [ ]:
# Простая итоговая таблица для отправки
final_table = top_compare[[
    'target_rf', 'tariff_group', 'rank', 'report_month_str', 'agr_id',
    'excel_commission_monthly', 'cft_calc_summ',
    'excel_gross', 'delta_calc', 'company_name', 'inn', 'filial_rf', 'tariff_raw',
]].copy()

final_table = final_table.rename(columns={
    'excel_commission_monthly': 'excel_commission',
    'cft_calc_summ': 'lake_commission',
})

# стабильный порядок: РФ → тариф → rank → месяц
final_table['tariff_group'] = pd.Categorical(
    final_table['tariff_group'], categories=TARIFF_ORDER, ordered=True
)
final_table['report_month_str'] = pd.Categorical(
    final_table['report_month_str'], categories=MONTHS, ordered=True
)
final_table = final_table.sort_values(
    ['target_rf', 'tariff_group', 'rank', 'report_month_str', 'agr_id']
).reset_index(drop=True)

# компактный вид: только нужные колонки
final_simple = final_table[[
    'target_rf', 'tariff_group', 'report_month_str', 'agr_id',
    'excel_commission', 'lake_commission',
]].copy()

print(f'final_simple rows={len(final_simple):,} | agr_id={final_simple["agr_id"].nunique()}')
print('Ожидание: до', len(TARGET_RFS) * len(TARIFF_ORDER) * TOP_N * len(MONTHS),
      '= РФ × тариф × TOP_N × месяцы')
display(final_simple)

# контроль: 5 agr на тариф×РФ, по 3 месяца
qc_cnt = (
    final_simple.groupby(['target_rf', 'tariff_group'], as_index=False)
    .agg(agr_n=('agr_id', 'nunique'), rows=('agr_id', 'size'))
)
display(qc_cnt)


## 6) Выгрузка всех строк озера (DOG_OPER) по выбранным agr_id

Комиссия в ЦФТ может быть разбита на несколько строк — выгружаем **сырые** операции, не только месячный итог.


In [ ]:
# Все сырые строки DOG_OPER по TOP agr_id
if raw_df.empty or not SELECTED_IDS:
    selected_raw = pd.DataFrame()
    print('WARN: нет raw_df или SELECTED_IDS — lake raw пустой')
else:
    sel_raw = raw_df.copy()
    sel_raw['id_agreement'] = sel_raw['id_agreement'].astype(str).str.replace(r'\.0$', '', regex=True)
    selected_raw = sel_raw[sel_raw['id_agreement'].isin(SELECTED_IDS)].copy()
    selected_raw['c_date_create'] = pd.to_datetime(selected_raw['c_date_create'], errors='coerce')
    selected_raw['report_month_str'] = selected_raw['c_date_create'].dt.strftime('%Y-%m')
    # атрибуты из Excel-отбора
    meta = top_agr_df[['target_rf', 'tariff_group', 'agr_id', 'rank', 'company_name', 'inn']].copy()
    meta['agr_id'] = meta['agr_id'].astype(str)
    selected_raw = selected_raw.merge(
        meta, left_on='id_agreement', right_on='agr_id', how='left'
    )
    selected_raw = selected_raw.sort_values(
        ['target_rf', 'tariff_group', 'rank', 'id_agreement', 'c_date_create']
    ).reset_index(drop=True)

print(f'selected_raw rows={len(selected_raw):,} | agr_id={selected_raw["id_agreement"].nunique() if len(selected_raw) else 0}')
if len(selected_raw):
    display(
        selected_raw.groupby(['target_rf', 'tariff_group', 'id_agreement'], as_index=False)
        .size().rename(columns={'size': 'lake_rows'}).head(40)
    )
    display(selected_raw.head(40))


## 7) Гипотеза «забыли НДС»: тариф `0`, Excel ≈ озеро (без ×1.22)

Если в озере должны быть суммы **с НДС**, а цифры **совпадают** с Excel (без НДС) — похоже, НДС не начислили.

Отбор: тариф `0`, по **5 agr_id на каждый РФ**, где хотя бы в одном месяце:
- `|lake_calc − excel| ≤ 0.01` (сырое совпадение),
- `excel > 0` (отсекаем 0=0),
- `|lake_calc − excel×1.22| > 0.01` (после НДС это уже расхождение).

Ранг: больше месяцев с таким паттерном, затем большая сумма Excel.


In [ ]:
VAT_FORGOT_TARIFF = '0'
MIN_EXCEL_COMM = COMPARE_ABS_TOL  # не берём нулевые комиссии

cmp0 = compare_df[compare_df['tariff_group'] == VAT_FORGOT_TARIFF].copy()
cmp0['delta_raw'] = cmp0['cft_calc_summ'] - cmp0['excel_commission_monthly']
cmp0['abs_delta_raw'] = cmp0['delta_raw'].abs()
cmp0['raw_match'] = (
    (cmp0['abs_delta_raw'] <= COMPARE_ABS_TOL)
    & (cmp0['excel_commission_monthly'].abs() > MIN_EXCEL_COMM)
    & (cmp0['cft_rows'] > 0)
)
# совпали без НДС, но НЕ совпали с НДС → кандидат на «забыли НДС»
cmp0['vat_forgot_month'] = cmp0['raw_match'] & (cmp0['abs_delta_calc'] > COMPARE_ABS_TOL)

vat_forgot_score = (
    cmp0.groupby(['target_rf', 'agr_id'], as_index=False)
    .agg(
        vat_forgot_months=('vat_forgot_month', 'sum'),
        raw_match_months=('raw_match', 'sum'),
        excel_max=('excel_commission_monthly', 'max'),
        lake_max=('cft_calc_summ', 'max'),
        company_name=('company_name', 'first'),
        inn=('inn', 'first'),
        filial_rf=('filial_rf', 'first'),
        tariff_raw=('tariff_raw', 'first'),
    )
)
vat_forgot_score = vat_forgot_score[vat_forgot_score['vat_forgot_months'] > 0].copy()

vat_forgot_top_rows = []
for rf in TARGET_RFS:
    pool = vat_forgot_score[vat_forgot_score['target_rf'] == rf].copy()
    pool = pool.sort_values(
        ['vat_forgot_months', 'excel_max', 'agr_id'],
        ascending=[False, False, True],
    ).head(TOP_N)
    pool['rank'] = range(1, len(pool) + 1)
    vat_forgot_top_rows.append(pool)
    print(
        f'{rf} | tariff 0 | vat_forgot candidates={len(vat_forgot_score[vat_forgot_score["target_rf"]==rf])}, '
        f'top={len(pool)}'
    )
    if len(pool) < TOP_N:
        print(f'WARN: {rf}: нашлось только {len(pool)} agr_id с паттерном «Excel≈озеро без НДС»')

vat_forgot_agr_df = (
    pd.concat(vat_forgot_top_rows, ignore_index=True) if vat_forgot_top_rows else pd.DataFrame()
)
VAT_FORGOT_IDS = sorted(set(vat_forgot_agr_df['agr_id'].astype(str))) if len(vat_forgot_agr_df) else []
print('VAT_FORGOT_IDS:', len(VAT_FORGOT_IDS), VAT_FORGOT_IDS)
display(vat_forgot_agr_df)

if len(vat_forgot_agr_df):
    vat_forgot_by_month = cmp0.merge(
        vat_forgot_agr_df[['target_rf', 'agr_id', 'rank', 'vat_forgot_months']],
        on=['target_rf', 'agr_id'],
        how='inner',
    ).copy()
    vat_forgot_simple = vat_forgot_by_month[[
        'target_rf', 'report_month_str', 'rank', 'agr_id',
        'excel_commission_monthly', 'cft_calc_summ', 'excel_gross',
        'delta_raw', 'delta_calc', 'vat_forgot_month',
        'company_name', 'inn', 'filial_rf',
    ]].rename(columns={
        'excel_commission_monthly': 'excel_commission',
        'cft_calc_summ': 'lake_commission',
    })
    vat_forgot_simple['report_month_str'] = pd.Categorical(
        vat_forgot_simple['report_month_str'], categories=MONTHS, ordered=True
    )
    vat_forgot_simple = vat_forgot_simple.sort_values(
        ['target_rf', 'rank', 'report_month_str', 'agr_id']
    ).reset_index(drop=True)
else:
    vat_forgot_by_month = pd.DataFrame()
    vat_forgot_simple = pd.DataFrame(columns=[
        'target_rf', 'report_month_str', 'agr_id', 'excel_commission', 'lake_commission',
    ])

print('vat_forgot_simple:')
display(vat_forgot_simple)

# сырые строки озера по этим agr_id
if raw_df.empty or not VAT_FORGOT_IDS:
    vat_forgot_raw = pd.DataFrame()
else:
    vr = raw_df.copy()
    vr['id_agreement'] = vr['id_agreement'].astype(str).str.replace(r'\.0$', '', regex=True)
    vat_forgot_raw = vr[vr['id_agreement'].isin(VAT_FORGOT_IDS)].copy()
    vat_forgot_raw['c_date_create'] = pd.to_datetime(vat_forgot_raw['c_date_create'], errors='coerce')
    meta_vf = vat_forgot_agr_df[['target_rf', 'agr_id', 'rank', 'company_name', 'inn']].copy()
    meta_vf['agr_id'] = meta_vf['agr_id'].astype(str)
    vat_forgot_raw = vat_forgot_raw.merge(
        meta_vf, left_on='id_agreement', right_on='agr_id', how='left'
    )
    vat_forgot_raw = vat_forgot_raw.sort_values(
        ['target_rf', 'rank', 'id_agreement', 'c_date_create']
    ).reset_index(drop=True)

print(f'vat_forgot_raw rows={len(vat_forgot_raw):,}')
if len(vat_forgot_raw):
    display(vat_forgot_raw.head(30))


In [ ]:
# Сохранение итогов
out_final = OUT_DIR / 'final_top5_agr_excel_vs_lake_by_month.xlsx'
out_raw_sel = OUT_DIR / 'lake_dog_oper_selected_agr_all_rows.xlsx'
out_compare = OUT_DIR / 'compare_excel_vs_cft_by_month.xlsx'
out_vat_forgot = OUT_DIR / 'tariff0_vat_forgot_excel_eq_lake.xlsx'
out_vat_forgot_raw = OUT_DIR / 'tariff0_vat_forgot_lake_raw_rows.xlsx'

with pd.ExcelWriter(out_final, engine='openpyxl') as writer:
    final_simple.to_excel(writer, sheet_name='simple', index=False)
    final_table.to_excel(writer, sheet_name='with_vat_delta', index=False)
    top_agr_df.to_excel(writer, sheet_name='top5_agr_list', index=False)
    for rf in TARGET_RFS:
        slug = rf_slug(rf)[:25]
        part = final_simple[final_simple['target_rf'] == rf]
        part.to_excel(writer, sheet_name=slug, index=False)

selected_raw.to_excel(out_raw_sel, index=False)

cand_summary = (
    agr_score.groupby(['target_rf', 'tariff_group'], as_index=False)
    .agg(
        candidates=('agr_id', 'nunique'),
        with_disc=('score', lambda s: int((s > COMPARE_ABS_TOL).sum())),
    )
)
rf_summary = (
    top_agr_df.groupby(['target_rf', 'tariff_group'], as_index=False)
    .agg(selected=('agr_id', 'nunique'), max_score=('score', 'max'))
)
selection_log = cand_summary.merge(rf_summary, on=['target_rf', 'tariff_group'], how='left')

with pd.ExcelWriter(out_compare, engine='openpyxl') as writer:
    compare_df.sort_values(['target_rf', 'tariff_group', 'agr_id', 'report_month_str']).to_excel(
        writer, sheet_name='compare_all_candidates', index=False
    )
    agr_score.sort_values(['target_rf', 'tariff_group', 'score'], ascending=[True, True, False]).to_excel(
        writer, sheet_name='agr_scores', index=False
    )
    selection_log.to_excel(writer, sheet_name='selection_log', index=False)
    final_simple.to_excel(writer, sheet_name='final_simple', index=False)

with pd.ExcelWriter(out_vat_forgot, engine='openpyxl') as writer:
    vat_forgot_simple.to_excel(writer, sheet_name='simple_by_month', index=False)
    vat_forgot_agr_df.to_excel(writer, sheet_name='top5_agr', index=False)
    for rf in TARGET_RFS:
        slug = rf_slug(rf)[:25]
        part = vat_forgot_simple[vat_forgot_simple['target_rf'] == rf]
        part.to_excel(writer, sheet_name=slug, index=False)

vat_forgot_raw.to_excel(out_vat_forgot_raw, index=False)

# per-RF удобные файлы
for rf in TARGET_RFS:
    slug = rf_slug(rf)
    out_xlsx = OUT_DIR / f'{slug}_top5_mismatch.xlsx'
    with pd.ExcelWriter(out_xlsx, engine='openpyxl') as writer:
        final_simple[final_simple['target_rf'] == rf].to_excel(writer, sheet_name='simple', index=False)
        final_table[final_table['target_rf'] == rf].to_excel(writer, sheet_name='with_vat_delta', index=False)
        top_agr_df[top_agr_df['target_rf'] == rf].to_excel(writer, sheet_name='top5_agr', index=False)
        if len(selected_raw):
            selected_raw[selected_raw['target_rf'] == rf].to_excel(
                writer, sheet_name='lake_raw_rows', index=False
            )
        if len(vat_forgot_simple):
            vat_forgot_simple[vat_forgot_simple['target_rf'] == rf].to_excel(
                writer, sheet_name='vat_forgot_t0', index=False
            )
        if len(vat_forgot_raw):
            vat_forgot_raw[vat_forgot_raw['target_rf'] == rf].to_excel(
                writer, sheet_name='vat_forgot_t0_raw', index=False
            )
    print('Saved:', out_xlsx)

print('Saved:', out_final)
print('Saved:', out_raw_sel, '| rows=', len(selected_raw))
print('Saved:', out_compare)
print('Saved:', out_vat_forgot)
print('Saved:', out_vat_forgot_raw, '| rows=', len(vat_forgot_raw))
print('OUT_DIR:')
for p in sorted(OUT_DIR.glob('*')):
    print(' ', p.name)
